In [1]:
import datacube
import pandas as pd
import geopandas as gpd
from datetime import datetime, timedelta
from odc.geo import Geometry
from dea_tools.bandindices import calculate_indices
import os
import glob
from concurrent.futures import ProcessPoolExecutor, as_completed

import sys
sys.path.append(os.path.abspath('./Tools/dea_tools'))
from classification import collect_training_data

In [2]:
# check the right up-to-data function was imported
collect_training_data?

Signature:
collect_training_data(
    gdf: geopandas.geodataframe.GeoDataFrame,
    dc_query: dict[str, typing.Any],
    ncpus: int = 1,
    chunksize: int | None = 1,
    return_coords: bool = False,
    feature_func: <built-in function callable> = None,
    field: str = None,
    zonal_stats: Optional[str] = None,
    clean: bool = True,
    fail_threshold: float = 0.05,
    fail_ratio: float = 0.5,
    max_retries: int = 2,
    time_field: Optional[str] = None,
    time_delta: Optional[datetime.timedelta] = None,
) -> pandas.core.frame.DataFrame
Docstring:
This function provides methods for gathering training/validation data from the ODC over
geometries stored within a geopandas geodataframe. The function will return a
pandas.DataFrame where the index contains class labels and the columns contain 
feature values generated by a user-defined `feature_func`.  

- In the instance where ncpus > 1, the function will automatically run in parallel.
- Zonal statistics are supported where the

In [3]:
samples = gpd.read_file('test_sample_locations.geojson')
samples=samples.to_crs('EPSG:3577')
samples = samples.rename(columns={"class": "label"}) # name 'class' gives issues later on
samples.head()

,label,geometry
0,1,POINT (1853295 -2953725)
1,1,POINT (1246755 -3990375)
2,1,POINT (775455 -2537445)
3,1,POINT (1960095 -2988915)
4,1,POINT (1509135 -3261075)


In [4]:
# add time column - for simplicity, only one date for all

samples['date'] = datetime(2024,7,1)
samples.head(3)


,label,geometry,date
0,1,POINT (1853295 -2953725),2024-07-01
1,1,POINT (1246755 -3990375),2024-07-01
2,1,POINT (775455 -2537445),2024-07-01


In [5]:
def feat_func(dc_query):
    dc = datacube.Datacube(app='feature_layers')
    ds_ard = dc.load(**dc_query)
    
    ds_ard = calculate_indices(
        ds_ard,
        index=['NDVI', 'NDWI', 'MNDWI'],
        drop=False,
        collection='ga_ls_3',
        verbose=False
    )
    
    return ds_ard#.squeeze(dim="time")

In [6]:
training_data = collect_training_data(
    gdf=samples.iloc[:100],
    dc_query={
        'product':'ga_ls9c_ard_3',
        'measurements':["nbart_blue","nbart_green","nbart_red","nbart_nir","nbart_swir_1","nbart_swir_2","oa_fmask"],
        'output_crs': "epsg:3577",
        'skip_broken_datasets':True
    },
    ncpus=40, 
    return_coords=True, 
    feature_func=feat_func,
    field='label',
    clean=False,
    time_field='date',
    time_delta=timedelta(days=15), # this can be set to include a whole year
)

  0%|          | 0/100 [00:00<?, ?it/s]

Percentage of possible fails after run 1 = 0.0 %
Returning data without cleaning
Output shape:  (293, 16)


In [7]:
training_data

,time,y,x,nbart_blue,nbart_green,nbart_red,nbart_nir,nbart_swir_1,nbart_swir_2,oa_fmask,NDVI,NDWI,MNDWI,x_coord,y_coord
label,,,,,,,,,,,,,,,
1,2024-06-18 23:53:13.093918,-2953725.0,1853295.0,332,693,638,239,424,423,5,-0.454960,0.487124,0.240824,1853295.0,-2953725.0
1,2024-07-04 23:53:21.380074,-2953725.0,1853295.0,307,674,608,378,313,308,5,-0.233266,0.281369,0.365755,1853295.0,-2953725.0
1,2024-06-17 00:08:25.316572,-3990375.0,1246755.0,218,721,472,199,85,92,5,-0.406855,0.567391,0.789082,1246755.0,-3990375.0
1,2024-06-26 00:02:06.303661,-3990375.0,1246755.0,259,707,460,265,97,115,5,-0.268966,0.454733,0.758706,1246755.0,-3990375.0
1,2024-07-03 00:08:27.659292,-3990375.0,1246755.0,196,626,415,190,89,70,5,-0.371901,0.534314,0.751049,1246755.0,-3990375.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,2024-07-07 01:16:20.047856,-1452915.0,-53445.0,348,570,500,871,414,275,1,0.270605,-0.208883,0.158537,-53445.0,-1452915.0
1,2024-07-07 01:16:43.966881,-1452915.0,-53445.0,350,566,496,860,413,272,1,0.268437,-0.206171,0.156282,-53445.0,-1452915.0
1,2024-06-21 01:15:41.759795,-1241535.0,93435.0,781,1275,1381,1312,366,225,1,-0.025622,-0.014302,0.553931,93435.0,-1241535.0


### try with time submitted in query only 

In [8]:
samples_2 = samples.drop(columns=['date'])
samples_2.head(3)

,label,geometry
0,1,POINT (1853295 -2953725)
1,1,POINT (1246755 -3990375)
2,1,POINT (775455 -2537445)


In [9]:
training_data_2 = collect_training_data(
    gdf=samples.iloc[:100],
    dc_query={
        'product':'ga_ls9c_ard_3',
        'measurements':["nbart_blue","nbart_green","nbart_red","nbart_nir","nbart_swir_1","nbart_swir_2","oa_fmask"],
        'output_crs': "epsg:3577",
        'time':(datetime(2024,7,1)-timedelta(days=15),datetime(2024,7,1)+timedelta(days=15)),
        'skip_broken_datasets':True
    },
    ncpus=40, 
    return_coords=True, 
    feature_func=feat_func,
    field='label',
    clean=False, 
)

  0%|          | 0/100 [00:00<?, ?it/s]

Percentage of possible fails after run 1 = 0.0 %
Returning data without cleaning
Output shape:  (293, 16)


In [10]:
training_data_2

,time,y,x,nbart_blue,nbart_green,nbart_red,nbart_nir,nbart_swir_1,nbart_swir_2,oa_fmask,NDVI,NDWI,MNDWI,x_coord,y_coord
label,,,,,,,,,,,,,,,
1,2024-06-18 23:53:13.093918,-2953725.0,1853295.0,332,693,638,239,424,423,5,-0.454960,0.487124,0.240824,1853295.0,-2953725.0
1,2024-07-04 23:53:21.380074,-2953725.0,1853295.0,307,674,608,378,313,308,5,-0.233266,0.281369,0.365755,1853295.0,-2953725.0
1,2024-06-17 00:08:25.316572,-3990375.0,1246755.0,218,721,472,199,85,92,5,-0.406855,0.567391,0.789082,1246755.0,-3990375.0
1,2024-06-26 00:02:06.303661,-3990375.0,1246755.0,259,707,460,265,97,115,5,-0.268966,0.454733,0.758706,1246755.0,-3990375.0
1,2024-07-03 00:08:27.659292,-3990375.0,1246755.0,196,626,415,190,89,70,5,-0.371901,0.534314,0.751049,1246755.0,-3990375.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,2024-07-07 01:16:20.047856,-1452915.0,-53445.0,348,570,500,871,414,275,1,0.270605,-0.208883,0.158537,-53445.0,-1452915.0
1,2024-07-07 01:16:43.966881,-1452915.0,-53445.0,350,566,496,860,413,272,1,0.268437,-0.206171,0.156282,-53445.0,-1452915.0
1,2024-06-21 01:15:41.759795,-1241535.0,93435.0,781,1275,1381,1312,366,225,1,-0.025622,-0.014302,0.553931,93435.0,-1241535.0
